(function-maps)=
# Function Map Application

The {ref}`data tree structures <data-tree>` allow for the application of specific functions on each of their nodes.
While it is easy to apply a single function on all nodes of a data tree, CRACE provides facilities to "map" different functions to subtrees or individual nodes.
At the same time, CRACE also supports "storing" functions in a dedicated registry, making it possible to identify them with their name, too.

Nodes can be identified via their path or their name (the last segment of their path).
Functions can either be specified as callable objects

In [ ]:
import xarray as xr
import numpy as np
import crace as ce

In [ ]:
dset = xr.Dataset({"var": (["x", "y"], np.zeros((3, 2)))})
tree = xr.DataTree.from_dict(
    {
        "/": None,
        "/a": dset.copy(),
        "/a/1": dset.copy(),
        "/a/2": dset.copy(),
        "/b": dset.copy(),
        "/b/1": dset.copy(),
        "/b/1/x": dset.copy(),
        "/b/2": dset.copy(),
        "/b/3": None,
        "/c": dset.copy(),
    }
)
tree

## Mapping Impact Functions

CRACE has no general notion of "impact functions".
The distinction to other functions is only chosen in function and parameter names, as applying an "impact function" (as opposed to other types of functions) might have specific implications, as shown further below.
For the purpose of CRACE, "impact functions" are functions that transform datasets by conforming to the following signature:

```python
def func(arg: xarray.Dataset) -> xarray.Dataset: ...
```

Inside {py:class}`crace.Engine`, they are applied to the {py:attr}`~crace.Engine.hazard` data tree (which is isomorphic to {py:attr}`~crace.Engine.exposure`) to yield the pixel- or point-wise ratio of impacted exposure.
The result is then multiplied with {py:attr}`~crace.Engine.exposure` to yield the impact.

```{hint}
Impact functions may be registered with a name using the {py:func}`~crace.impact_function` decorator.
```

{py:class}`~crace.Engine` performs this application via {py:func}`~crace.map_impact_function`, which will usually not be called in user code.
However, the application of the impact functions in {py:class}`~crace.Engine` thus follow the exact rules of this function, which is why we investigate it here.

We define a function map as dictionary with keys of type `str` or {py:class}`crace.FuncType`, and values of type callable.

In [ ]:
@ce.impact_function(name="my_func")
def func(x: xr.Dataset) -> xr.Dataset:
    return x


def times_three(x: xr.Dataset) -> xr.Dataset:
    return x + 3


func_map = {
    "a": lambda x: x + 1,
    "1": lambda x: x + 2,
    "/b/2": times_three,
    ce.FuncType.leaf: "my_func",
}

Following the rules set by {py:func}`~crace.map_impact_function`, the tree nodes will be matched with the following keys:

```{list-table}
:widths: "auto"
:header-rows: 1

*   - `tree` Node
    - `func_map` Key
    - Matching Rule
    - Result
*   - `"/"`
    - *no match*
    - 
    - No dataset (no match)
*   - `"/a"`
    - `"a"`
    - Matched by name
    - 1
*   - `"/a/1"`
    - `"1"`
    - Matched by name
    - 2
*   - `"/a/2"`
    - `"a"`
    - Matched by parent
    - 1
*   - `"/b"`
    - *no match*
    - 
    - No dataset (no match)
*   - `"/b/1"`
    - `"1"`
    - Matched by name
    - 2
*   - `"/b/1/x"`
    - `"1"`
    - Matched by parent
    - 2
*   - `"/b/2"`
    - `"/b/2"`
    - Matched by path
    - 3
*   - `"/b/3"`
    - `ce.FuncType.leaf`
    - Matched via leaf property
    - No dataset (no original)
*   - `"/c"`
    - `ce.FuncType.leaf`
    - Matched via leaf property
    - 0
```

In [ ]:
import crace as ce


result = ce.map_impact_function(tree, func_map)
result

## Mapping Aggregate Functions

The application of "aggregate functions" differs slightly from that of "impact functions".
When applying an impact function to a data tree, we want to ensure that it is easy to specify how *all* nodes are transformed.
Obiously, the impact function should be applied to the entire hazard data.
This is different from aggregates, as these might only be required at specific levels or nodes of the tree.
For example, one might want to apply different impact f